# 04 - Collaborative Filtering

This data has no ratings anywhere — only purchase counts and a binary reordered flag — so
classic explicit-rating CF (SVD, etc.) doesn't fit. The right tool for this shape of data is
ALS for implicit feedback (Hu, Koren & Volinsky), via the `implicit` library: purchase counts
become confidence weights in how sure we are a user likes a product, not preference scores.

Two things about this dataset specifically shape how the model is set up:

- **Reorder rate is 0.59** — most of what shows up in a user's next basket is something
  they've already bought before. `implicit`'s `recommend()` defaults to filtering out
  previously-purchased items, which would work directly against this task, so that's turned
  off explicitly.
- **User/product IDs aren't contiguous** — a sparse matrix needs row/column indices starting
  at 0, so IDs get mapped to matrix positions and back. That mapping logic is tested
  separately in `src/collaborative_filtering.py` against known purchase counts, since a
  silent mismatch here would corrupt every recommendation without throwing an error.

Logic in `src/collaborative_filtering.py`. Needs `pip install implicit` if it's not already
in the environment.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd

from data_processing import load_raw_data
from eda import build_transactions
from baseline_models import get_eval_users, evaluate_model
from collaborative_filtering import build_user_item_matrix, ALSModel

## Load data

In [2]:
processed_path = Path.cwd().parent / "data" / "processed" / "transactions.parquet"

if processed_path.exists():
    txn = pd.read_parquet(processed_path)
else:
    data = load_raw_data()
    txn = build_transactions(data)

print(txn.shape)

(33819106, 15)


In [3]:
eval_users = get_eval_users(txn)
print("users with a labeled train basket:", len(eval_users))

users with a labeled train basket: 131209


## Build the user-item matrix


Use a sparse CSR matrix rather than a dense user-item matrix. With approximately
206k users and 50k products, a dense representation would require nearly
10 billion cells and be impractical to allocate. The sparse representation
stores only observed user-product interactions, making it substantially more
memory-efficient for collaborative filtering.


In [4]:
matrix, user_to_idx, product_to_idx, idx_to_product = build_user_item_matrix(txn)
print("matrix shape:", matrix.shape)
print("non-zero entries:", matrix.nnz)
print("density: {:.5f}%".format(100 * matrix.nnz / (matrix.shape[0] * matrix.shape[1])))

matrix shape: (206209, 49677)
non-zero entries: 13307953
density: 0.12991%


## Fit ALS


Initialize ALS with 50 latent factors and 15 iterations as a practical starting
configuration for the collaborative-filtering model. The lower factor count
provides sufficient model capacity while keeping the initial configuration
tractable for the user-item matrix size.

Use alpha=40 as the initial confidence-weighting parameter for purchase counts.
This value will be evaluated against the baseline, with alpha treated
as a tunable parameter based on observed recommendation performance.


In [5]:
als_model = ALSModel(factors=50, regularization=0.01, alpha=40.0, iterations=15).fit(txn)

C:\Users\shubh\AppData\Roaming\Python\Python314\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

In [6]:
sample_user = eval_users.iloc[0]["user_id"]
print(f"recs for user {sample_user}:", als_model.recommend(sample_user, n=10))

recs for user 1: [np.int32(37710), np.int32(31651), np.int32(38928), np.int32(6184), np.int32(41400), np.int32(32455), np.int32(22802), np.int32(196), np.int32(13176), np.int32(46149)]


## Evaluate

Same harness as baseline — Precision@10 / Recall@10 against each user's actual train basket —
so this is a direct, apples-to-apples comparison against the Personalized Frequency baseline
(0.284 precision / 0.330 recall on the full dataset).

In [7]:
K = 10
als_results = evaluate_model(lambda uid: als_model.recommend(uid, n=K), eval_users, k=K)
als_results

{'k': 10,
 'n_users_evaluated': 131209,
 'precision_at_k': 0.06551989573885939,
 'recall_at_k': 0.0981054421813323}

In [8]:
pd.DataFrame([als_results], index=["ALS"]).to_csv(
    Path.cwd().parent / "data" / "processed" / "als_results.csv"
)

## Alpha sensitivity

Precision/recall came back well below Personalized Frequency at `alpha=40`. That's not
surprising on its own — grocery reordering is driven by habit far more than by the kind of
similarity signal ALS is built to find, and this is a well-documented outcome on this exact
dataset (the published Instacart competition writeups found the same thing: frequency/reorder
features beat pure collaborative filtering here). `alpha` controls how strongly purchase
counts get converted into confidence weights, and it's the one parameter most likely to move
the result meaningfully without a full tuning pass, so it's worth checking a couple of other
values before drawing a conclusion.

factors and iterations are left as they were — changing more than one variable at a time
would make it unclear what's actually driving any difference.

In [9]:
alpha_results = {}

for alpha_value in [15.0, 40.0, 100.0]:
    model = ALSModel(factors=50, regularization=0.01, alpha=alpha_value, iterations=15).fit(txn)
    alpha_results[f"alpha={alpha_value:.0f}"] = evaluate_model(
        lambda uid: model.recommend(uid, n=K), eval_users, k=K
    )

pd.DataFrame(alpha_results).T

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

,k,n_users_evaluated,precision_at_k,recall_at_k
alpha=15,10.0,131209.0,0.089410,0.121091
alpha=40,10.0,131209.0,0.065520,0.098105
alpha=100,10.0,131209.0,0.044162,0.074812


In [10]:
pd.DataFrame(alpha_results).T.to_csv(
    Path.cwd().parent / "data" / "processed" / "als_alpha_comparison.csv"
)

### Findings

The collaborative filtering analysis evaluates ALS as the next modeling layer after the baseline models. The model is trained using implicit purchase feedback, where purchase counts represent confidence rather than explicit preference ratings. The user-item matrix contains **206,209 users and 49,677 products**, with approximately **13.3M non-zero interactions** and a density of only **0.12991%**, confirming that a sparse representation is appropriate.

* **ALS underperforms the personalized baseline.** At `alpha=40`, ALS achieves **Precision@10 = 0.0655** and **Recall@10 = 0.0981**, substantially below Personalized Frequency baseline of **0.284 precision and 0.330 recall**.

* **Lower alpha improves ALS performance.** The sensitivity analysis shows that `alpha=15` performs best among the tested values, reaching **0.0894 Precision@10** and **0.1211 Recall@10**. Increasing alpha to `40` and `100` progressively reduces performance, indicating that stronger confidence weighting of purchase counts does not improve recommendations under the current evaluation setup.

* **Purchase-history signals remain substantially stronger.** The results are consistent with the EDA finding that approximately **59% of purchases are reorders**. This indicates that customer-specific historical purchasing behavior is a stronger signal for predicting the next basket than latent collaborative patterns alone.

* **ALS still provides complementary information.** Although ALS does not provide a competitive standalone reorder predictor, its recommendations can capture product relationships that are not explicitly represented by a customer's purchase frequency. This makes ALS potentially useful as a **discovery or cross-sell signal** rather than as the primary ranking mechanism.

* **Modeling implication:** The results support moving forward with a **hybrid recommendation approach**, retaining Personalized Frequency as the primary reorder signal while using ALS as a complementary signal. This allows the next phase to test whether collaborative information provides incremental value beyond the strong historical-purchase baseline.
